In [1]:
# 1. Atualizando a biblioteca financeira (garante que vamos puxar os dados mais recentes)
!pip install -q -U yfinance pandas

import yfinance as yf
import pandas as pd

print("🌐 Conectando ao banco de dados financeiro...")

# 2. Definindo a empresa da B3. 
# Regra de Ouro: No yfinance, ações brasileiras precisam do sufixo ".SA"
ticker = "WEGE3.SA"  
empresa = yf.Ticker(ticker)

# 3. Puxando os dados fundamentais
# .balance_sheet traz o Balanço Patrimonial
# .financials traz o DRE (Demonstrativo de Resultados)
balanco_patrimonial = empresa.balance_sheet
dre = empresa.financials

print(f"\n✅ Dados extraídos com sucesso para: {ticker}")

# 4. Limpando a exibição para o nosso Valuation
print("\n📊 Prévia do Balanço Patrimonial (Últimos anos):")
display(balanco_patrimonial.head(10)) # Mostra as 10 primeiras linhas (Ativos, Caixa, etc.)

print("\n📈 Prévia do DRE (Receitas e Lucros):")
display(dre.head(10))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 73.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 re

,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Treasury Shares Number,1.622025e+06,1.780620e+06,2.083696e+06,1.302592e+06,NaN
Ordinary Shares Number,4.195696e+09,4.195537e+09,4.195234e+09,4.196015e+09,NaN
Share Issued,4.197318e+09,4.197318e+09,4.197318e+09,4.197318e+09,NaN
Total Debt,5.437975e+09,4.418355e+09,3.391960e+09,4.009322e+09,NaN
Tangible Book Value,1.463220e+10,1.938357e+10,1.587084e+10,1.331078e+10,NaN
Invested Capital,2.200801e+10,2.579946e+10,2.017715e+10,1.829449e+10,NaN
Working Capital,9.524444e+09,1.176709e+10,1.034262e+10,9.390333e+09,NaN
Net Tangible Assets,1.463220e+10,1.938357e+10,1.587084e+10,1.331078e+10,NaN
Capital Lease Obligations,8.471530e+08,8.231180e+08,5.568990e+08,5.496300e+08,NaN
Common Stock Equity,1.741718e+10,2.220422e+10,1.734208e+10,1.483480e+10,NaN



📈 Prévia do DRE (Receitas e Lucros):


,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Tax Effect Of Unusual Items,1.140097e+08,4.842633e+07,7.228352e+07,4.013558e+07,NaN
Tax Rate For Calcs,1.684000e-01,2.010000e-01,1.097000e-01,1.647000e-01,NaN
Normalized EBITDA,8.651314e+09,8.646793e+09,6.709981e+09,5.515876e+09,NaN
Total Unusual Items,6.770170e+08,2.409270e+08,6.589200e+08,2.436890e+08,NaN
Total Unusual Items Excluding Goodwill,6.770170e+08,2.409270e+08,6.589200e+08,2.436890e+08,NaN
Net Income From Continuing Operation Net Minority Interest,6.376219e+09,6.042593e+09,5.731670e+09,4.208084e+09,NaN
Reconciled Depreciation,1.001296e+09,8.124850e+08,6.280420e+08,5.655570e+08,NaN
Reconciled Cost Of Revenue,2.712248e+10,2.517310e+10,2.170274e+10,2.120924e+10,NaN
EBITDA,9.328331e+09,8.887720e+09,7.368901e+09,5.759565e+09,NaN
EBIT,8.327035e+09,8.075235e+09,6.740859e+09,5.194008e+09,NaN


In [2]:
import numpy as np

print("⚙️ Construindo o Motor de Valuation (DCF)...")

# 1. Puxando o Fluxo de Caixa e Dados Complementares
fluxo_caixa = empresa.cashflow
info = empresa.info

# Extraindo métricas chave (usando o último ano disponível como base)
# Tratamento de erro com .get() caso a empresa não tenha a métrica preenchida perfeitamente
caixa_total = info.get('totalCash', 0)
divida_total = info.get('totalDebt', 0)
acoes_em_circulacao = info.get('sharesOutstanding', 1) 

# Simplificação: Pegando o Fluxo de Caixa Livre (Free Cash Flow) mais recente
try:
    fcf_atual = fluxo_caixa.loc['Free Cash Flow'].iloc[0]
except KeyError:
    fcf_atual = fluxo_caixa.loc['Operating Cash Flow'].iloc[0] - fluxo_caixa.loc['Capital Expenditure'].iloc[0]

print(f"💰 Fluxo de Caixa Livre Base: R$ {fcf_atual:,.2f}")
print(f"🏦 Caixa Total: R$ {caixa_total:,.2f} | 📉 Dívida Total: R$ {divida_total:,.2f}")

# 2. Parâmetros do DCF (Premissas Iniciais)
# Aqui é onde o seu LLM vai entrar no futuro para ajustar essas taxas lendo o risco nas notícias!
taxa_crescimento_anos_1_a_5 = 0.08  # Crescimento projetado de 8% ao ano
taxa_crescimento_perpetuidade = 0.03 # Crescimento na perpetuidade (após ano 5) de 3%
wacc = 0.11                         # Taxa de Desconto (Custo Médio Ponderado de Capital) de 11%

# 3. Projetando os Fluxos Futuros (5 anos)
fluxos_projetados = []
fcf_projetado = fcf_atual

for ano in range(1, 6):
    fcf_projetado *= (1 + taxa_crescimento_anos_1_a_5)
    fluxos_projetados.append(fcf_projetado)

# 4. Trazendo a Valor Presente (VP)
vp_fluxos = [f / ((1 + wacc) ** i) for i, f in enumerate(fluxos_projetados, 1)]
soma_vp_fluxos = sum(vp_fluxos)

# 5. Valor Terminal (A perpetuidade trazida a valor presente)
valor_terminal = (fluxos_projetados[-1] * (1 + taxa_crescimento_perpetuidade)) / (wacc - taxa_crescimento_perpetuidade)
vp_valor_terminal = valor_terminal / ((1 + wacc) ** 5)

# 6. Cálculo Final do Preço Justo da Ação
enterprise_value = soma_vp_fluxos + vp_valor_terminal
equity_value = enterprise_value + caixa_total - divida_total
preco_justo = equity_value / acoes_em_circulacao

print("\n" + "="*50)
print(f"🎯 VALUATION CONCLUÍDO: {ticker}")
print("="*50)
print(f"Valor da Empresa (Enterprise Value): R$ {enterprise_value:,.2f}")
print(f"Valor para o Acionista (Equity Value): R$ {equity_value:,.2f}")
print(f"📊 PREÇO JUSTO POR AÇÃO PROJETADO: R$ {preco_justo:.2f}")
print(f"💲 PREÇO ATUAL DE MERCADO: R$ {info.get('currentPrice', 0):.2f}")
print("="*50)

⚙️ Construindo o Motor de Valuation (DCF)...
💰 Fluxo de Caixa Livre Base: R$ 3,759,712,000.00
🏦 Caixa Total: R$ 7,279,864,832.00 | 📉 Dívida Total: R$ 5,437,975,040.00

🎯 VALUATION CONCLUÍDO: WEGE3.SA
Valor da Empresa (Enterprise Value): R$ 59,537,239,314.25
Valor para o Acionista (Equity Value): R$ 61,379,129,106.25
📊 PREÇO JUSTO POR AÇÃO PROJETADO: R$ 14.63
💲 PREÇO ATUAL DE MERCADO: R$ 50.62


In [3]:
import datetime

print(f"📡 Acionando o Radar de Notícias para {ticker}...\n")

noticias_brutas = empresa.news
pacote_texto_llm = ""

if noticias_brutas:
    print(f"✅ Encontramos notícias recentes. Extraindo o sinal...\n")
    print("="*90)
    
    for i, noticia in enumerate(noticias_brutas[:5]):
        # 1. Acessando o "cofre" (a chave 'content' que descobrimos no debug)
        conteudo = noticia.get('content', {})
        
        # 2. Extraindo os dados da sala secundária
        titulo = conteudo.get('title', 'Sem título')
        resumo = conteudo.get('summary', 'Sem resumo disponível')
        
        # 3. Tratando a Data (O formato ISO do Yahoo)
        data_pub_bruta = conteudo.get('pubDate', '')
        if data_pub_bruta:
            try:
                # Transforma '2026-04-02T16:13:53Z' em '02/04/2026'
                data_obj = datetime.datetime.strptime(data_pub_bruta, '%Y-%m-%dT%H:%M:%SZ')
                data_pub = data_obj.strftime('%d/%m/%Y')
            except ValueError:
                data_pub = data_pub_bruta
        else:
            data_pub = "Data Recente"
        
        # 4. Acessando o provedor (que está em uma terceira sala: content -> provider -> displayName)
        provedor = conteudo.get('provider', {}).get('displayName', 'Fonte Desconhecida')
        
        # Mostrando na tela para o Analista
        print(f"[{i+1}] 📅 {data_pub} | 🏢 {provedor}")
        print(f"🗞️ Manchete: {titulo}")
        print(f"📝 Resumo: {resumo}")
        print("-" * 90)
        
        # Empacotando para o Cérebro da IA
        pacote_texto_llm += f"Notícia {i+1} ({data_pub} - {provedor}): {titulo}. Resumo: {resumo}\n\n"

    print("\n📦 Pacote de Texto Formulado com Sucesso para o LLM!")
    
else:
    print("⚠️ Nenhuma notícia recente encontrada.")

# Imprimindo a visão limpa final
print("\n🤖 VISÃO DO LLM (Input Bruto que será enviado ao Nemotron):")
print(pacote_texto_llm)

📡 Acionando o Radar de Notícias para WEGE3.SA...

✅ Encontramos notícias recentes. Extraindo o sinal...

[1] 📅 02/04/2026 | 🏢 Simply Wall St.
🗞️ Manchete: How The WEG (BOVESPA:WEGE3) Investment Story Is Resetting After JPMorgan’s Target Cut
📝 Resumo: WEG is back in focus after a reset in expectations, with the key talking point being the shift in JPMorgan’s price target from R$50 to R$49 compared with fair value estimates of R$54.26. Bullish and bearish analysts are reading that small change very differently, with some highlighting headroom relative to fair value and others stressing what the cut says about conviction after the Q4 miss. As you read on, you will see how this evolving narrative could shape the way you track WEG from...
------------------------------------------------------------------------------------------
[2] 📅 06/03/2026 | 🏢 Simply Wall St.
🗞️ Manchete: How The WEG (BOVESPA:WEGE3) Investment Story Is Shifting After JPMorgan’s Target Cut
📝 Resumo: WEG is back in focus

In [4]:
import re

print("🧠 Iniciando Módulo de Fusão: LLM + Valuation...\n")

# 1. A Engenharia de Prompt Restritiva
# Nós "engessamos" a IA para que ela atue estritamente como um Analista Financeiro (Persona)
prompt_sistema = f"""
Você é um Analista Chefe de Equity Research de um fundo Quantitativo.
A taxa de crescimento base (Base Growth Rate) projetada para a empresa WEGE3.SA é de 8.0% (0.08) ao ano.

Leia as notícias recentes abaixo e avalie o sentimento do mercado, riscos e oportunidades.
Contexto de Notícias:
{pacote_texto_llm}

Sua tarefa: Ajustar a taxa de crescimento base. Se as notícias forem muito positivas (ex: forte crescimento de receita, novas parcerias de peso), aumente a taxa (ex: 0.09 ou 0.12). Se houver alertas graves de risco (ex: corte agressivo de preço-alvo por grandes bancos, perda de margem), reduza a taxa (ex: 0.07 ou 0.05).

Responda ESTRITAMENTE neste formato, sem adicionar mais nada:
JUSTIFICATIVA: [Sua análise em uma frase]
TAXA_AJUSTADA: [Apenas o número decimal, usando ponto]
"""

print("📝 Prompt gerado e enviado para a IA. Aguardando processamento...")

# ==========================================
# 2. CHAMADA DA IA (SIMULAÇÃO DE INFERÊNCIA PARA O PIPELINE)
# Aqui entraria o seu modelo Nemotron. Para fins de montagem do pipeline hoje, 
# estamos simulando a saída exata que um modelo bem treinado daria após ler nosso pacote:
resposta_llm = """
JUSTIFICATIVA: Embora haja pressão nas margens e um leve corte de target pelo JPMorgan, o crescimento agressivo de receita de 25.5% e a parceria estratégica com a Petrobras em energia eólica indicam uma resiliência e expansão superiores ao cenário base conservador.
TAXA_AJUSTADA: 0.115
"""
# ==========================================

print(f"\n🤖 RESPOSTA DA IA:\n{resposta_llm}")

# 3. Extração Numérica (Expressão Regular)
# O código python precisa "pescar" apenas o número que a IA gerou no meio do texto
match = re.search(r"TAXA_AJUSTADA:\s*([0-9.]+)", resposta_llm)

if match:
    nova_taxa_crescimento = float(match.group(1))
    print("-" * 50)
    print(f"✅ Extração bem sucedida!")
    print(f"📉 Taxa Base: 8.0%")
    print(f"📈 Nova Taxa Definida pela IA: {nova_taxa_crescimento * 100}%")
    print("-" * 50)
else:
    print("⚠️ Erro: O LLM não retornou o número no formato esperado.")
    nova_taxa_crescimento = 0.08 # Fallback de segurança para a taxa base

🧠 Iniciando Módulo de Fusão: LLM + Valuation...

📝 Prompt gerado e enviado para a IA. Aguardando processamento...

🤖 RESPOSTA DA IA:

JUSTIFICATIVA: Embora haja pressão nas margens e um leve corte de target pelo JPMorgan, o crescimento agressivo de receita de 25.5% e a parceria estratégica com a Petrobras em energia eólica indicam uma resiliência e expansão superiores ao cenário base conservador.
TAXA_AJUSTADA: 0.115

--------------------------------------------------
✅ Extração bem sucedida!
📉 Taxa Base: 8.0%
📈 Nova Taxa Definida pela IA: 11.5%
--------------------------------------------------


In [5]:
print("🚀 RECALCULANDO O VALUATION COM O CÉREBRO DA IA...\n")

# Usando a nova taxa extraída pelo LLM (0.115) em vez da taxa base engessada (0.08)
fluxos_projetados_ia = []
fcf_projetado_ia = fcf_atual

# 1. Projetando os 5 anos com a visão otimista de mercado lida nas notícias
for ano in range(1, 6):
    fcf_projetado_ia *= (1 + nova_taxa_crescimento) # A IA assumiu o controle aqui!
    fluxos_projetados_ia.append(fcf_projetado_ia)

# 2. Trazendo a Valor Presente (VP)
vp_fluxos_ia = [f / ((1 + wacc) ** i) for i, f in enumerate(fluxos_projetados_ia, 1)]
soma_vp_fluxos_ia = sum(vp_fluxos_ia)

# 3. Valor Terminal
valor_terminal_ia = (fluxos_projetados_ia[-1] * (1 + taxa_crescimento_perpetuidade)) / (wacc - taxa_crescimento_perpetuidade)
vp_valor_terminal_ia = valor_terminal_ia / ((1 + wacc) ** 5)

# 4. Cálculo Final de Fechamento
enterprise_value_ia = soma_vp_fluxos_ia + vp_valor_terminal_ia
equity_value_ia = enterprise_value_ia + caixa_total - divida_total
preco_justo_ia = equity_value_ia / acoes_em_circulacao

print("="*65)
print("🎯 VALUATION DEFINITIVO: IA + DCF (PIPELINE CONCLUÍDO)")
print("="*65)
print(f"📉 Preço Justo Antigo (Matemática Cega - Cresc. 8%): R$ 14.63")
print(f"📈 NOVO PREÇO JUSTO PROJETADO (Visão LLM - Cresc. {nova_taxa_crescimento*100}%): R$ {preco_justo_ia:.2f}")
print(f"💲 PREÇO ATUAL DE MERCADO: R$ {info.get('currentPrice', 0):.2f}")
print("="*65)
print("\nO seu modelo deixou de ser uma planilha estática e passou a 'ler' o mercado. Parabéns pela arquitetura!")

🚀 RECALCULANDO O VALUATION COM O CÉREBRO DA IA...

🎯 VALUATION DEFINITIVO: IA + DCF (PIPELINE CONCLUÍDO)
📉 Preço Justo Antigo (Matemática Cega - Cresc. 8%): R$ 14.63
📈 NOVO PREÇO JUSTO PROJETADO (Visão LLM - Cresc. 11.5%): R$ 16.78
💲 PREÇO ATUAL DE MERCADO: R$ 50.62

O seu modelo deixou de ser uma planilha estática e passou a 'ler' o mercado. Parabéns pela arquitetura!


In [6]:
# 1. Instalando a biblioteca moderna de leitura de PDFs
!pip install -q pypdf

import os
from pypdf import PdfReader

print("📄 Iniciando o Módulo Inside-Out: Leitor de Relatórios Oficiais...\n")

# O diretório padrão onde o Kaggle salva os arquivos que você faz upload
caminho_pdf = "/kaggle/working/weg_release.pdf"
texto_pdf_extraido = ""

if os.path.exists(caminho_pdf):
    print("✅ Arquivo 'weg_release.pdf' encontrado! Iniciando mineração de dados...")
    try:
        leitor = PdfReader(caminho_pdf)
        
        # 🎯 Tática Sniper: Lendo apenas as 3 primeiras páginas para poupar o cérebro da IA (Mensagem da Administração)
        num_paginas = min(3, len(leitor.pages)) 
        
        for i in range(num_paginas):
            pagina = leitor.pages[i]
            texto_pdf_extraido += pagina.extract_text() + "\n"
            
        print(f"✅ Extração de {num_paginas} páginas concluída com sucesso.")
        
    except Exception as e:
        print(f"⚠️ Erro ao tentar ler o PDF: {e}")
else:
    print("⚠️ ALERTA: Arquivo 'weg_release.pdf' não encontrado no diretório local.")
    print("💡 Ativando contingência: injetando texto simulado do último Release para manter o pipeline ativo...\n")
    texto_pdf_extraido = """
    MENSAGEM DA ADMINISTRAÇÃO E DESTAQUES: 
    A WEG apresentou desempenho sólido no trimestre. A Receita Operacional Líquida (ROL) cresceu impulsionada pela forte demanda no mercado externo, 
    especialmente em equipamentos eletroeletrônicos industriais de ciclo longo para os setores de óleo, gás e mineração. 
    A margem EBITDA manteve-se resiliente e em patamares históricos, demonstrando nossa disciplina de custos e ganhos de escala. 
    Continuamos focados na transição energética global, anunciando a construção de novas plantas na Europa e ampliação das operações na América do Norte. 
    Apesar das pressões inflacionárias pontuais e do cenário macroeconômico global desafiador, mantemos perspectiva de crescimento contínuo 
    e rentabilidade elevada para os próximos exercícios.
    """

# 2. Limpeza de Dados (Sanitização)
# Removendo quebras de linha excessivas que confundem o LLM
texto_pdf_extraido = " ".join(texto_pdf_extraido.split())

print("📦 VISÃO DO LLM - Trecho do Relatório Oficial (Visão da Diretoria):")
print(texto_pdf_extraido[:300] + " [...]")

# 3. A GRANDE FUSÃO: Unindo as Notícias (Outside) com o Relatório Oficial (Inside)
pacote_final_conhecimento = f"""
==== VISÃO DO MERCADO (NOTÍCIAS RECENTES) ====
{pacote_texto_llm}

==== VISÃO DA EMPRESA (RELATÓRIO OFICIAL) ====
{texto_pdf_extraido}
"""

print("\n🚀 FUSÃO CONCLUÍDA! O 'pacote_final_conhecimento' está pronto para a IA Real.")

📄 Iniciando o Módulo Inside-Out: Leitor de Relatórios Oficiais...

⚠️ ALERTA: Arquivo 'weg_release.pdf' não encontrado no diretório local.
💡 Ativando contingência: injetando texto simulado do último Release para manter o pipeline ativo...

📦 VISÃO DO LLM - Trecho do Relatório Oficial (Visão da Diretoria):
MENSAGEM DA ADMINISTRAÇÃO E DESTAQUES: A WEG apresentou desempenho sólido no trimestre. A Receita Operacional Líquida (ROL) cresceu impulsionada pela forte demanda no mercado externo, especialmente em equipamentos eletroeletrônicos industriais de ciclo longo para os setores de óleo, gás e mineração. [...]

🚀 FUSÃO CONCLUÍDA! O 'pacote_final_conhecimento' está pronto para a IA Real.


In [7]:
# 1. Instalando a NOVA biblioteca oficial da IA do Google
!pip install -q -U google-genai

from google import genai
from kaggle_secrets import UserSecretsClient
import re

print("🧠 Iniciando Módulo de Fusão: Conectando à IA Real (Nova API Gemini)...\n")

try:
    # 2. Resgatando a chave do cofre do Kaggle
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    
    # Nova inicialização do Cliente da API
    client = genai.Client(api_key=api_key)
    
    # 3. O Prompt de Engenharia
    prompt_sistema = f"""
    Você é um Analista Chefe de Equity Research de um fundo Quantitativo.
    A taxa de crescimento base (Base Growth Rate) projetada para a empresa WEGE3.SA é de 8.0% (0.08) ao ano.

    Leia o pacote de conhecimento abaixo (Visão de Mercado e Visão da Empresa) e avalie o sentimento, riscos e oportunidades.
    
    CONTEXTO DE DADOS:
    {pacote_final_conhecimento}

    Sua tarefa: Ajustar a taxa de crescimento base. 
    - Se o contexto for muito positivo (forte expansão, lucro alto, visão otimista), aumente a taxa (ex: 0.09 a 0.15). 
    - Se houver alertas de risco ou pessimismo, reduza a taxa (ex: 0.07 a 0.02).

    Responda ESTRITAMENTE neste formato exato, com duas linhas:
    JUSTIFICATIVA: [Sua análise resumida em até 2 frases detalhando o porquê do ajuste]
    TAXA_AJUSTADA: [Apenas o número decimal, usando ponto. Ex: 0.115]
    """

    print("📡 Enviando o Dossiê para a IA. Aguardando a análise financeira...")
    
    # 4. A Chamada Real para a Nuvem (Atualizada para o novo padrão)
    # Usando a família atualizada de modelos flash
    resposta = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt_sistema
    )
    
    resposta_texto = resposta.text.strip()
    
    print("\n" + "="*60)
    print("🤖 RESPOSTA OFICIAL DA IA:")
    print("="*60)
    print(resposta_texto)
    print("="*60 + "\n")

    # 5. Extração Numérica e Matemática
    match = re.search(r"TAXA_AJUSTADA:\s*([0-9.]+)", resposta_texto)

    if match:
        nova_taxa_crescimento_real = float(match.group(1))
        print(f"✅ Extração bem sucedida!")
        print(f"📉 Taxa Base: 8.0%")
        print(f"📈 Nova Taxa Definida pela IA: {nova_taxa_crescimento_real * 100:.2f}%")
        
        # Recalculando o DCF com o novo Cérebro
        wacc = 0.11
        taxa_perpetuidade = 0.03
        fcf_projetado = fcf_atual
        fluxos = []
        
        for ano in range(1, 6):
            fcf_projetado *= (1 + nova_taxa_crescimento_real)
            fluxos.append(fcf_projetado)
            
        vp_fluxos = sum([f / ((1 + wacc) ** i) for i, f in enumerate(fluxos, 1)])
        v_terminal = (fluxos[-1] * (1 + taxa_perpetuidade)) / (wacc - taxa_perpetuidade)
        vp_terminal = v_terminal / ((1 + wacc) ** 5)
        
        preco_justo_final = (vp_fluxos + vp_terminal + caixa_total - divida_total) / acoes_em_circulacao
        
        print(f"\n🎯 VALUATION FINAL ATUALIZADO PELA IA: R$ {preco_justo_final:.2f}")

    else:
        print("⚠️ Erro: A IA não retornou a taxa no formato numérico esperado.")

except Exception as e:
    print(f"🚨 Erro de Execução: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible

In [8]:
import yfinance as yf
import datetime
import re
import pandas as pd
from google import genai
from kaggle_secrets import UserSecretsClient

print("🏭 Iniciando Linha de Produção: Screener de Ações via LLM...\n")

# 1. Preparando o Cérebro
user_secrets = UserSecretsClient()
client = genai.Client(api_key=user_secrets.get_secret("GEMINI_API_KEY"))

# 2. Nossa Lista de Alvos
acoes_alvo = ['WEGE3.SA', 'VALE3.SA']
resultados_screener = []

# 3. A Função Mestra (O Motor Empacotado)
def analisar_acao_automatica(ticker):
    print(f"🔍 Analisando {ticker}...")
    try:
        empresa = yf.Ticker(ticker)
        noticias_brutas = empresa.news
        pacote_noticias = ""
        
        # Coletando Notícias
        if noticias_brutas:
            for i, noticia in enumerate(noticias_brutas[:4]):
                conteudo = noticia.get('content', {})
                titulo = conteudo.get('title', 'Sem título')
                resumo = conteudo.get('summary', '')
                pacote_noticias += f"- {titulo}: {resumo}\n"
        else:
            pacote_noticias = "Nenhuma notícia relevante encontrada nos últimos dias."

        # O Prompt Escalável
        prompt = f"""
        Atue como Analista Quantitativo. Analise as notícias recentes da empresa {ticker}:
        {pacote_noticias}
        
        A taxa de crescimento neutra é 5.0% (0.05). 
        Ajuste essa taxa com base no otimismo ou pessimismo das notícias (mínimo 0.01, máximo 0.15).
        Responda ESTRITAMENTE:
        JUSTIFICATIVA: [Uma frase]
        TAXA: [Numero decimal, ex: 0.08]
        """
        
        # Chamada da IA
        resposta = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        resposta_texto = resposta.text.strip()
        
        # Extração
        match = re.search(r"TAXA:\s*([0-9.]+)", resposta_texto)
        taxa_ia = float(match.group(1)) if match else 0.05
        
        # Pegando o Preço Atual para comparação
        preco_atual = empresa.info.get('currentPrice', 0)
        
        return {
            "Ticker": ticker,
            "Preço Atual (R$)": round(preco_atual, 2),
            "Taxa Projetada IA": f"{taxa_ia * 100:.1f}%",
            "Tese da IA": resposta_texto.split('\n')[0].replace('JUSTIFICATIVA: ', '')
        }
        
    except Exception as e:
        print(f"⚠️ Erro ao processar {ticker}: {e}")
        return None

# 4. Rodando a Linha de Montagem
for acao in acoes_alvo:
    resultado = analisar_acao_automatica(acao)
    if resultado:
        resultados_screener.append(resultado)

# 5. O Relatório Final do Gestor (DataFrame)
print("\n" + "="*80)
print("📊 RELATÓRIO DO FUNDO: SCREENER DE OPORTUNIDADES")
print("="*80)
df_relatorio = pd.DataFrame(resultados_screener)
display(df_relatorio)

🏭 Iniciando Linha de Produção: Screener de Ações via LLM...

🔍 Analisando WEGE3.SA...
🔍 Analisando VALE3.SA...

📊 RELATÓRIO DO FUNDO: SCREENER DE OPORTUNIDADES


,Ticker,Preço Atual (R$),Taxa Projetada IA,Tese da IA
0,WEGE3.SA,50.62,7.0%,Apesar de pequenos cortes no preço-alvo e pres...
1,VALE3.SA,83.09,13.0%,"O forte otimismo do mercado, com analistas ele..."


In [9]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import re
from google import genai
import os

# --- CONFIGURAÇÃO DA PÁGINA ---
st.set_page_config(page_title="AutoValuation LLM", page_icon="📈", layout="wide")

st.title("📈 AutoValuation: Motor Quantitativo com IA")
st.markdown("Bem-vindo ao Screener de Valuation. Digite os Tickers da B3 para a Inteligência Artificial analisar o sentimento do mercado e projetar a taxa de crescimento.")

# --- BARRA LATERAL (INPUTS) ---
with st.sidebar:
    st.header("⚙️ Configurações")
    chave_api = st.text_input("Cole sua API Key do Gemini:", type="password")
    tickers_input = st.text_input("Tickers (separados por vírgula):", "WEGE3.SA, VALE3.SA, PETR4.SA")
    btn_analisar = st.button("🚀 Iniciar Análise Automática")

# --- MOTOR DE ANÁLISE (A Função que você já testou) ---
def analisar_acao(ticker, client):
    try:
        empresa = yf.Ticker(ticker)
        noticias = empresa.news
        pacote_noticias = "".join([f"- {n.get('content', {}).get('title', '')}\n" for n in noticias[:4]]) if noticias else "Sem notícias."

        prompt = f"""
        Atue como Analista Quantitativo. Analise as notícias de {ticker}:\n{pacote_noticias}\n
        A taxa de crescimento neutra é 5.0%. Ajuste-a com base nas notícias (mín 0.01, máx 0.15).
        Responda ESTRITAMENTE:\nJUSTIFICATIVA: [Uma frase]\nTAXA: [Numero decimal]
        """
        
        resposta = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        match = re.search(r"TAXA:\s*([0-9.]+)", resposta.text)
        taxa_ia = float(match.group(1)) if match else 0.05
        
        return {
            "Ticker": ticker,
            "Preço (R$)": round(empresa.info.get('currentPrice', 0), 2),
            "Crescimento (IA)": f"{taxa_ia * 100:.1f}%",
            "Tese de Investimento": resposta.text.split('\n')[0].replace('JUSTIFICATIVA: ', '')
        }
    except Exception as e:
        return {"Ticker": ticker, "Erro": str(e)}

# --- EXECUÇÃO VISUAL ---
if btn_analisar:
    if not chave_api:
        st.warning("⚠️ Por favor, insira a sua API Key do Google Gemini na barra lateral.")
    else:
        # Mostra um spinner de carregamento bonito na tela
        with st.spinner("🤖 A IA está lendo o mercado... Isso pode levar alguns segundos."):
            client = genai.Client(api_key=chave_api)
            lista_tickers = [t.strip() for t in tickers_input.split(",")]
            resultados = []
            
            for t in lista_tickers:
                resultados.append(analisar_acao(t, client))
            
            df = pd.DataFrame(resultados)
            
            st.success("✅ Análise Concluída com Sucesso!")
            st.dataframe(df, use_container_width=True) # Desenha a tabela interativa na tela
            
            st.markdown("---")
            st.markdown("*Disclaimer: Este é um projeto de Engenharia de Dados e IA. Não constitui recomendação de compra ou venda de ativos.*")

Writing app.py
